# 11 — Feature Engineering and Preprocessing

## 1. Objective and scope boundary

This notebook freezes the feature policy only. It decides which features may proceed into feature-engineering experiments, without creating those features or performing preprocessing.

```text
Notebook 10: What does the data tell us?
        ↓
Notebook 11 policy freeze: Which features are allowed to proceed?
        ↓
Next work: How are approved features created?
```

> The policy freeze establishes feature eligibility before transformation. This prevents preprocessing code from implicitly deciding which columns are allowed into the model.

An approved candidate is allowed into Phase 2 design; it is **not** guaranteed to be a final model feature.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from urban_ops.features.policy import (
    PolicyStatus,
    feature_policy_table,
    load_feature_policy,
    validate_feature_policy_evidence,
)

POLICY_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_baseline.yaml'

def file_state(path):
    return (sha256(path.read_bytes()).hexdigest(), path.stat().st_mtime_ns)

latest_pointer = PROJECT_ROOT / 'data/splits/resolution_risk/latest.json'
latest_payload = json.loads(latest_pointer.read_text(encoding='utf-8'))
configured_run = Path(latest_payload['run_path'])
split_run = configured_run if configured_run.is_absolute() else PROJECT_ROOT / configured_run
governed_inputs = [
    latest_pointer,
    split_run / 'train.parquet',
    split_run / 'validation.parquet',
    split_run / 'test.parquet',
    split_run / 'split_metadata.json',
    split_run / 'split_rules_snapshot.yaml',
]
governed_inputs.extend(sorted((PROJECT_ROOT / 'reports/11_split_aware_eda').rglob('*')))
governed_inputs = tuple(path for path in governed_inputs if path.is_file())
before_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}

pd.Series({'governed_input_count': len(before_states), 'split_id': latest_payload['split_id']})

governed_input_count                                   54
split_id                20260806T135114Z_9d945cb2da0eecfc
dtype: object

## 2. Notebook 10 handoff authority

Notebook 10 completed its analysis and reconciliation, but reported `model_ready: False`. That is correct: EDA describes the data and recommends possibilities, but it does not freeze which representations are permitted to proceed.

The authority order is: Step 4 leakage and prediction-time availability; Notebook 10 leakage audit; Notebook 10's newer `eda_status`; train-only structure evidence; then the older `baseline_decision`. A statistically interesting feature can remain excluded or conditional because governance takes priority over descriptive EDA patterns.

In [2]:
policy = load_feature_policy(POLICY_PATH)
policy_table = feature_policy_table(policy)
display(pd.Series(policy.notebook_10_handoff, name='Notebook 10 handoff'))
display(pd.DataFrame(policy.authority_hierarchy).sort_values('priority'))

split_id            20260806T135114Z_9d945cb2da0eecfc
integrity                                        PASS
reconciliation                                   PASS
step_9a_decision                             COMPLETE
model_ready                                     False
Name: Notebook 10 handoff, dtype: object

,priority,authority,source
0,1,Step 4 leakage and prediction-time availability,docs/leakage_policy.md
1,2,Notebook 10 leakage audit,reports/11_split_aware_eda/tables/leakage_audi...
2,3,Notebook 10 EDA statuses,notebooks/10_split_aware_eda.ipynb
3,4,"Notebook 10 missingness, cardinality, and outl...",reports/11_split_aware_eda/
4,5,Notebook 10 baseline recommendations,reports/11_split_aware_eda/tables/baseline_fea...


## 3. Feature-policy decision rules

A feature receives `phase_2_allowed = True` only after all of these gates pass:

1. available at the complaint-creation prediction moment;
2. safe under Step 4 leakage governance;
3. non-null with usable train variation;
4. the approved representation rather than an alternative or redundant form; and
5. no unresolved policy or temporal-generalization blocker.

Target-rate differences alone can never grant Phase 2 eligibility.

## 4. Approved first-pass candidates

The four preferred temporal representations are frozen as candidates. Their common source is retained, but no calendar field is derived during this work.

In [3]:
approved = policy_table.loc[policy_table['policy_status'].eq('APPROVED_CANDIDATE')]
display(approved[['feature_name', 'source_column', 'policy_status', 'eda_status', 'reason', 'phase_2_allowed']])
assert tuple(approved['feature_name']) == policy.feature_creation_allow_list

,feature_name,source_column,policy_status,eda_status,reason,phase_2_allowed
1,created_hour,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,CANDIDATE,Simple creation-time-safe operational grouping...,True


## 5. Alternative and redundant representations

Names and numeric calendar codes carry the same essential information for day and month, so the first baseline selects one representation. Quarter and week-of-year overlap with the preferred month representation and remain under redundancy review.

In [4]:
representation_review = policy_table.loc[policy_table['policy_status'].isin([
    'ALTERNATIVE_REPRESENTATION', 'REVIEW_REDUNDANCY', 'REVIEW'
])]
display(representation_review[['feature_name', 'policy_status', 'preferred_counterpart', 'redundancy_status', 'reason', 'phase_2_allowed']])

,feature_name,policy_status,preferred_counterpart,redundancy_status,reason,phase_2_allowed
5,created_day_name,ALTERNATIVE_REPRESENTATION,created_day_of_week,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,ALTERNATIVE_REPRESENTATION,created_month,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,REVIEW,NaN,NONE,"Creation-time safe, but current EDA does not j...",False


## 6. Conditional features

`created_year` may encode time progression or regime rather than a durable operational pattern. Geography remains unresolved because the current authority does not prove creation-time availability and immutability. Missingness strategies and apparent target associations do not resolve that uncertainty.

In [5]:
conditional = policy_table.loc[policy_table['policy_status'].eq('CONDITIONAL')]
display(conditional[['feature_name', 'prediction_time_status', 'leakage_status', 'reason', 'phase_2_allowed']])
assert not conditional['phase_2_allowed'].any()

,feature_name,prediction_time_status,leakage_status,reason,phase_2_allowed
10,created_year,AVAILABLE,SAFE,May encode temporal progression or regime rath...,False
11,borough,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
12,location_type,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
13,incident_zip,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
14,latitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
15,longitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False


## 7. Excluded features

Exclusions retain explicit reasons: leakage/post-creation/target-derived, identifier, all-null, or zero variance. `unique_key` remains available only for traceability. `due_date` remains blocked because its prediction-time availability and mutability are unproven.

In [6]:
excluded = policy_table.loc[policy_table['policy_status'].str.startswith('EXCLUDE_')]
display(excluded[['feature_name', 'policy_status', 'prediction_time_status', 'leakage_status', 'variation_status', 'reason', 'phase_2_allowed']])
assert not excluded['phase_2_allowed'].any()

,feature_name,policy_status,prediction_time_status,leakage_status,variation_status,reason,phase_2_allowed
16,unique_key,EXCLUDE_IDENTIFIER,AVAILABLE,BLOCKED,HAS_VARIATION,Identifier is retained for traceability only a...,False
17,descriptor_2,EXCLUDE_ALL_NULL,UNRESOLVED,CONDITIONAL,ALL_NULL,Training evidence identifies the field as enti...,False
18,agency,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
19,agency_name,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
20,complaint_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
21,descriptor,EXCLUDE_ZERO_VARIANCE,UNRESOLVED,CONDITIONAL,ZERO_VARIANCE,Training evidence shows no useful variation; e...,False
22,open_data_channel_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Training evidence shows one fixed channel valu...,False
23,closed_date,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,HAS_VARIATION,Closure outcome is unavailable at complaint cr...,False
24,due_date,EXCLUDE_LEAKAGE,UNRESOLVED,BLOCKED,HAS_VARIATION,Target input whose creation-time timing and mu...,False
25,status,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,ZERO_VARIANCE,Final mutable outcome status is post-creation ...,False


## 8. Frozen feature policy

This is the complete required policy table. The YAML remains the machine-readable source of truth; this view is generated from the validated loader.

In [7]:
required_columns = [
    'feature_name', 'source_column', 'policy_status', 'prediction_time_status',
    'leakage_status', 'eda_status', 'redundancy_status', 'reason', 'phase_2_allowed'
]
display(policy_table[required_columns])
assert policy_table['feature_name'].is_unique
assert policy_table[required_columns].notna().all().all()

,feature_name,source_column,policy_status,prediction_time_status,leakage_status,eda_status,redundancy_status,reason,phase_2_allowed
0,created_date,created_date,SOURCE_ONLY,AVAILABLE,SAFE,REVIEW,SOURCE_FOR_DERIVATIONS,Required to derive temporal features; the raw ...,False
1,created_hour,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Simple creation-time-safe operational grouping...,True
5,created_day_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,created_date,REVIEW,AVAILABLE,SAFE,REVIEW,NONE,"Creation-time safe, but current EDA does not j...",False


## 9. Policy validation

The validator reconciles the policy against Notebook 10's complete baseline inventory, Step 4/Notebook 10 leakage audit, all-null evidence, and zero-variance evidence. The final check also confirms every governed split artifact and Notebook 10 report has the same hash and modification time as at notebook start.

In [8]:
validate_feature_policy_evidence(policy)
after_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}
boundary_flags = [
    'transformations_implemented', 'missing_values_imputed',
    'rare_categories_grouped', 'encoder_fitted', 'scaler_fitted',
    'column_transformer_built', 'feature_matrix_created', 'model_trained'
]
policy_checks = pd.Series({
    'notebook_10_complete': policy.notebook_10_handoff['step_9a_decision'] == 'COMPLETE',
    'notebook_10_not_model_ready': policy.notebook_10_handoff['model_ready'] is False,
    'exact_feature_creation_allow_list': policy.feature_creation_allow_list == ('created_hour', 'created_day_of_week', 'created_month', 'is_weekend'),
    'only_approved_candidates_allowed': policy_table.loc[policy_table['phase_2_allowed'], 'policy_status'].eq('APPROVED_CANDIDATE').all(),
    'implementation_scope_respected': all(policy.implementation_boundary[flag] is False for flag in boundary_flags),
    'governed_sources_unchanged': before_states == after_states,
}, name='passed')
display(policy_checks)
assert policy_checks.all()

notebook_10_complete                 True
notebook_10_not_model_ready          True
exact_feature_creation_allow_list    True
only_approved_candidates_allowed     True
implementation_scope_respected       True
governed_sources_unchanged           True
Name: passed, dtype: bool

## 10. Completion decision

The policy freeze is complete when the policy and evidence reconcile, only approved candidates are allowed to proceed, and governed inputs remain unchanged. The data is still not model-ready because feature creation and every preprocessing decision remain separate reviewed work.

**STOP:** the next work is **Deterministic Feature Creation**. It is intentionally not implemented here.

In [9]:
pd.Series({
    'policy_decision': 'COMPLETE' if policy_checks.all() else 'NOT_COMPLETE',
    'model_ready': False,
    'feature_creation_allow_list': ', '.join(policy.feature_creation_allow_list),
    'next_work': 'Deterministic Feature Creation',
})

policy_decision                                                         COMPLETE
model_ready                                                                False
feature_creation_allow_list    created_hour, created_day_of_week, created_mon...
next_work                                         Deterministic Feature Creation
dtype: object